In [ ]:
# Empty for "ruff" extension

# Batch algorithm for Near-Duplicates Detection


In [ ]:
import sys
import pyspark.sql.functions as F
from pathlib import Path
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))

from src.spark_utils import get_spark
from src.datasets import read_synthetic

spark = get_spark()

%load_ext autoreload
%autoreload 2

In [ ]:
from src.near_duplicates import partitions_schema
import math


f = 64
p_min = 10
d = 20
probe_stats = []

for K in range(2, int(f // 2)):
    tables = partitions_schema(f=f, K=K, d=d,p_min=p_min)
    # Bits "covered" by the partitions (that are certainly equals)
    partition_bits = sum((table["r"] - K) * table["block_bits"] for table in tables[1])
    combos = tables[0]
    # Possible matches for each corpus row
    matches = combos * math.pow(2, d / partition_bits)
    probe_stats.append((K, matches))

df_stats = pd.DataFrame(probe_stats, columns=["K", "Avg matches per corpus row"]).set_index("K")
plt.figure(figsize=(15, 6))
ax = sns.lineplot(data=df_stats, markers=True)
plt.xticks(
    ticks=df_stats.index,
    labels=[str(el) for el in df_stats.index],
    rotation=45,
)
plt.xlabel("K values")
plt.yscale("log")
plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.7)
plt.tight_layout()
note = f"With {f}-bit fingerprints (f), corpus of 2^{d} elements (2^d), max 2^{p_min} candidates per probe (2^p_min)"
plt.figtext(
    0.5,  # X coord - middle
    -0.02,  # Y coord - under plot
    note,
    ha="center",
    va="top",
    fontsize=10,
    style="italic",
    bbox=dict(
        boxstyle="round,pad=0.5", facecolor="#f0f0f0", edgecolor="gray", lw=1
    ),
)
plt.show()

In [ ]:
from src.fingerprint import create_fingerprints
from src.near_duplicates import detect_near_duplicate_batch
from src.datasets import read_c5

# Other params are inherited from previous block
f = 64
K = 3
ID_COL = "ID"
INPUT_DATA_COL = "data"

input_df = read_c5()
fingerprints = create_fingerprints(df=input_df, model="bag_of_words", input_col=INPUT_DATA_COL, output_col="fingerprint", fingerprint_size=f, other=None).select(ID_COL, "fingerprint")
print(fingerprints.count())
fingerprints.cache()
corpus, query = fingerprints.randomSplit([0.8, 0.2])
result = detect_near_duplicate_batch(corpus, query, fingerprint_size=f, K=K)
result_size = result.count()
result_size

In [ ]:
duplicate_count = result.select("ID_corpus").distinct().count()
near_dup_ratio = "{:.2f}".format(duplicate_count / result_size * 100)
print(f"Near-Duplicate elements with maximum hamming distance {K}: {duplicate_count} out of {result_size} ({near_dup_ratio}%)")

In [ ]:
result_sample = result.limit(5)
near_dup_sample = input_df.alias("a").join(F.broadcast(result_sample).alias("b"), F.col(f"a.{ID_COL}") == F.col("b.ID_corpus")).select("b.ID_corpus", "b.ID_query", F.col(f"a.{INPUT_DATA_COL}").alias("corpus_text"))
near_dup_sample = input_df.alias("a").join(F.broadcast(near_dup_sample).alias("b"), F.col(f"a.{ID_COL}") == F.col("b.ID_query")).select("b.ID_corpus", "b.ID_query", "corpus_text", F.col(f"a.{INPUT_DATA_COL}").alias("query_text"))
near_dup_sample.show(truncate=True)

In [ ]:
spark.stop()